# Introduction
Natural Language Processing (NLP) helps computers understand and use human language, and it has the potential to improve many areas of our lives. But sometimes, these systems can be unfair because they reflect the biases found in the data they learn from. This can lead to stereotypes or discrimination against certain people or groups. To make NLP tools fair and respectful to everyone, it's important to find and fix these biases.

## Mitigating Bias in Algorithms
To make NLP models fairer, there are several effective strategies that can be used:

**Diverse Training Data:** Make sure the training data includes people from different backgrounds. You can also use data augmentation to fix imbalances.

**Bias Mitigation Techniques:** Apply methods like re-weighting samples, adversarial debiasing, or using word embeddings that have been adjusted to reduce bias.

**Ongoing Monitoring:** Regularly review and update models to catch and fix any new biases that may appear over time.

Example code for applying bias-corrected word embeddings:

In [ ]:
%pip install gensim

gensim is a popular NLP library used for working with word embeddings like Word2Vec, FastText, etc.

In [ ]:
import pandas as pd
import numpy as np
import nltk
from textblob import TextBlob
from gensim.models import KeyedVectors

Imports the nltk library: the Natural Language Toolkit.

Provides text processing libraries for classification, tokenization, stemming, tagging, parsing, and more.

from textblob import TextBlob
* Imports the TextBlob class from the textblob library.

Used for simple NLP tasks like sentiment analysis, noun phrase extraction, and translation.

from gensim.models import KeyedVectors

Imports KeyedVectors from gensim.models.

Used to load pre-trained word embedding models (like Word2Vec in .bin format).


In [ ]:
# Ensure the necessary NLTK data packages are downloaded
nltk.download('punkt')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

* nltk.download('punkt') downloads the Punkt tokenizer models used by NLTK.

* Punkt is a pre-trained model for sentence splitting and word tokenization.

* It must be downloaded once to allow text tokenization functions to work.

This code example demonstrates an end-to-end solution to identify and mitigate bias in an NLP model, focusing on text data analysis and bias correction using word embeddings. It includes data collection, preprocessing, bias detection, bias mitigation, and sentiment analysis.

In [ ]:
# Sample dataset
data = {
    'text': [
        "I love this product!",
        "This is terrible service.",
        "I'm so happy with the quality.",
        "This is the worst experience ever.",
        "As a company, you should not be in business.",
        "The service was not great, but the product quality is excellent."
    ],
    'gender': ['female', 'male', 'female', 'male', 'female', 'male']
}

df = pd.DataFrame(data)

* A Python dictionary data is created with two keys: text (user reviews) and gender (label for each text entry).

* Each value under text is a review/sentence.

* gender labels associate each review with either "male" or "female", possibly to test for bias or disparities.

In [ ]:
# Step 1: Audit representation of gender in the dataset
def audit_representation(df, column):
    counts = df[column].value_counts()
    return counts

Defines a function audit_representation that takes in:

df: the DataFrame,

column: the column to audit (in this case, gender).

value_counts() counts how many times each unique value (like "male" or "female") appears.



In [ ]:
# Output the representation of gender
gender_counts = audit_representation(df, 'gender')
print("Gender representation in the dataset:")
print(gender_counts)

Gender representation in the dataset:
gender
female    3
male      3
Name: count, dtype: int64


Calls the function on the 'gender' column.

Stores the result in gender_counts.

Prints the count of each gender to audit dataset balance. Output shows 3 males and 3 females – a balanced representation.

In [ ]:
# Step 2: Preprocess data (anonymize and clean text)
def preprocess_text(df, text_column):
    df[text_column] = df[text_column].apply(lambda x: x.lower())
    return df

df = preprocess_text(df, 'text')

A function preprocess_text is defined to clean the text data.

It lowercases all text to normalize casing for NLP processing (e.g., “Love” and “love” become the same).

lambda x: x.lower() is an inline function applied to each text string.

df = preprocess_text(df, 'text') -
The function is applied to the text column of the DataFrame to transform all reviews to lowercase.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from gensim.models import KeyedVectors

def load_word_vectors(file_path):
    try:
        word_vectors = KeyedVectors.load_word2vec_format(file_path, binary=True)
        print("Word vectors loaded successfully.")
        return word_vectors
    except Exception as e:
        print(f"Error loading word vectors: {e}")
        return None

# Provide path to decompressed binary model
word_vectors_path = '/content/drive/MyDrive/GoogleNews-vectors-negative300.bin'
word_vectors = load_word_vectors(word_vectors_path)


Word vectors loaded successfully.


This function loads pre-trained Word2Vec embeddings using Gensim.

load_word2vec_format(file_path, binary=True) loads binary .bin models (e.g., Google News).

If loading fails, an error is printed, and None is returned.

Tries to load vectors from a placeholder path.

In [ ]:
if word_vectors:
    # Define the gender direction vector
    def compute_gender_direction(word_vectors):
        try:
            gender_direction = word_vectors['woman'] - word_vectors['man']
            return gender_direction
        except KeyError as e:
            print(f"Error computing gender direction: {e}")
            return None

    gender_direction = compute_gender_direction(word_vectors)

    if gender_direction is not None:
        # Debias the word vectors
        def debias_word_vectors(word_vectors, gender_direction):
            # Create a writable copy of the vectors
            debiased_vectors = word_vectors.vectors.copy()
            debiased_vectors.flags['WRITEABLE'] = True

            for i, word in enumerate(word_vectors.index_to_key): # Use index_to_key to iterate by index
                vector = debiased_vectors[i]
                projection = np.dot(vector, gender_direction) * gender_direction
                debiased_vectors[i] -= projection

            # Update the original word_vectors object with the debiased vectors
            word_vectors.vectors = debiased_vectors


        debias_word_vectors(word_vectors, gender_direction)

if word_vectors:
Checks if word vectors loaded successfully.

Defines a function to compute the gender direction vector, which captures the directional difference between the words "woman" and "man".

This vector can help detect and mitigate gender bias in embeddings.

gender_direction = compute_gender_direction(word_vectors) -
Computes the gender direction.

Defines a function to remove gender bias from each word vector:

Projects each word vector onto the gender direction.

Subtracts that projection to make it neutral with respect to gender.

debias_word_vectors(word_vectors, gender_direction) - Applies the debiasing procedure across all vocabulary words.

In [ ]:
# Step 4: Sentiment analysis function
def analyze_sentiment(text):
    blob = TextBlob(text)
    return blob.sentiment.polarity

# Apply sentiment analysis
df['sentiment'] = df['text'].apply(analyze_sentiment)


Defines a function to perform sentiment analysis using TextBlob.

blob.sentiment.polarity returns a value between -1 and 1:

Negative = negative sentiment

Positive = positive sentiment

Near 0 = neutral

df['sentiment'] = df['text'].apply(analyze_sentiment) -
Applies sentiment analysis to each row of the text column.

Stores the result in a new column named sentiment.



Groups the dataset by the bias_column (in this case, 'gender') and computes average sentiment.

This helps visualize whether sentiment scores are skewed across gender.


In [ ]:
# Step 5: Bias mitigation in sentiment analysis
# For demonstration, we'll assume debiasing involves ensuring sentiment scores are not influenced by gender.

def mitigate_bias(df, text_column, sentiment_column, bias_column):
    try:
        avg_sentiment_by_gender = df.groupby(bias_column)[sentiment_column].mean()
        print("Average sentiment by gender before mitigation:")
        print(avg_sentiment_by_gender)

        # Example mitigation strategy: Adjust sentiment scores to remove gender bias influence
        gender_bias_adjustment = avg_sentiment_by_gender['male'] - avg_sentiment_by_gender['female']
        df[sentiment_column] = df.apply(lambda row: row[sentiment_column] - gender_bias_adjustment if row[bias_column] == 'male' else row[sentiment_column], axis=1)

        avg_sentiment_by_gender_after = df.groupby(bias_column)[sentiment_column].mean()
        print("Average sentiment by gender after mitigation:")
        print(avg_sentiment_by_gender_after)

        return df
    except Exception as e:
        print(f"Error mitigating bias: {e}")
        return df

df = mitigate_bias(df, 'text', 'sentiment', 'gender')

# Output results
print("Final dataset with mitigated bias and sentiment analysis:")
print(df)

Average sentiment by gender before mitigation:
gender
female    0.475000
male     -0.566667
Name: sentiment, dtype: float64
Average sentiment by gender after mitigation:
gender
female    0.475
male      0.475
Name: sentiment, dtype: float64
Final dataset with mitigated bias and sentiment analysis:
                                                text  gender  sentiment
0                               i love this product!  female   0.625000
1                          this is terrible service.    male   0.041667
2                     i'm so happy with the quality.  female   0.800000
3                 this is the worst experience ever.    male   0.041667
4       as a company, you should not be in business.  female   0.000000
5  the service was not great, but the product qua...    male   1.341667


def mitigate_bias(df, text_column, sentiment_column, bias_column):

Defines a function to mitigate bias in a DataFrame. It takes:

df: the dataset,

text_column: the column with text data (not used here but included for extensibility),

sentiment_column: the column with sentiment scores,

bias_column: the attribute by which bias is assessed (e.g., 'gender').


try:
    
     avg_sentiment_by_gender = df.groupby(bias_column)[sentiment_column].mean()

Groups the data by gender (or the specified bias attribute) and calculates the average sentiment score for each group. Displays the sentiment scores before applying any bias correction, so we can compare them later.

  gender_bias_adjustment = avg_sentiment_by_gender['male'] - avg_sentiment_by_gender['female']

Calculates the bias adjustment factor by taking the difference in average sentiment between males and females. If males have a lower sentiment, this value is negative and will be added to male scores to compensate.


        df[sentiment_column] = df.apply(
            lambda row: row[sentiment_column] - gender_bias_adjustment if row[bias_column] == 'male' else row[sentiment_column],
            axis=1
        )

Adjusts the sentiment scores:

For rows labeled as 'male', subtracts the gender_bias_adjustment from their sentiment.



        avg_sentiment_by_gender_after = df.groupby(bias_column)[sentiment_column].mean()
        print("Average sentiment by gender after mitigation:")
        print(avg_sentiment_by_gender_after)


Recalculates and prints average sentiment scores by gender after mitigation. The goal is to see that the adjusted scores are now equal or at least closer.

 return df

Returns the updated DataFrame with mitigated sentiment scores.

    except Exception as e:
        print(f"Error mitigating bias: {e}")
        return df

Handles any errors during mitigation. If something goes wrong, the function prints the error and returns the original DataFrame without changes.

df = mitigate_bias(df, 'text', 'sentiment', 'gender')

Executes the bias mitigation on the DataFrame using 'gender' as the bias attribute.

Displays the full DataFrame after mitigation, showing adjusted sentiment values that are no longer biased across gender groups.

